In [1]:
import json
from typing import Dict, Optional


def average_iter_metrics_for_batch(
    batch_size: int,
    file_path: str,
    tolerance: int = 4,
) -> Dict[str, Optional[float]]:
    """
    Compute average timing metrics from rows where BOTH:
    - abs(row['batch_size'] - batch_size) <= tolerance
    - abs(row['num_scheduled_tokens'] - batch_size) <= tolerance

    Returns averages for:
    latency_ms, model_compute_ms, expert_ffn_ms, attention_ms.
    Values are None if no numeric values are found for that metric.
    """
    metric_names = [
        "latency_ms",
        "model_compute_ms",
        "expert_ffn_ms",
        "attention_ms",
    ]

    sums = {metric: 0.0 for metric in metric_names}
    counts = {metric: 0 for metric in metric_names}
    matched_rows = 0

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            row = json.loads(line)

            row_batch_size = row.get("batch_size")
            row_tokens = row.get("num_scheduled_tokens")

            if row_batch_size is None or row_tokens is None:
                continue

            if (
                batch_size - row_batch_size <= tolerance
                and abs(batch_size - row_tokens) <= tolerance
            ):
                matched_rows += 1

                for metric in metric_names:
                    value = row.get(metric)
                    if isinstance(value, (int, float)):
                        sums[metric] += float(value)
                        counts[metric] += 1

    if matched_rows == 0:
        raise ValueError(
            "No rows matched. Check batch_size/tolerance, or verify the file content."
        )

    averages = {
        metric: (sums[metric] / counts[metric]) if counts[metric] else None
        for metric in metric_names
    }

    return {
        **averages,
        "matched_rows": float(matched_rows),
    }




In [2]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_40bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776647249336102073_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=40, file_path=sample_file_path)
result

{'latency_ms': 23.171858170838135,
 'model_compute_ms': 22.37037609290095,
 'expert_ffn_ms': 26.673177229192614,
 'attention_ms': 0.0,
 'matched_rows': 583.0}

In [3]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_16bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776647902399331951_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=16, file_path=sample_file_path)
result

{'latency_ms': 16.795476696473667,
 'model_compute_ms': 16.17129049137125,
 'expert_ffn_ms': 25.417028094363154,
 'attention_ms': 0.0,
 'matched_rows': 2035.0}

In [4]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_200bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776648129821772759_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=200, file_path=sample_file_path)
result

{'latency_ms': 36.960467176084165,
 'model_compute_ms': 33.20412411159939,
 'expert_ffn_ms': 27.045108747703058,
 'attention_ms': 6.1590153638963345,
 'matched_rows': 270.0}

In [5]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_140bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776648682840783640_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=140, file_path=sample_file_path)
result

{'latency_ms': 31.859840311971652,
 'model_compute_ms': 29.48852570202886,
 'expert_ffn_ms': 24.713860128302965,
 'attention_ms': 4.7913721861685215,
 'matched_rows': 588.0}

In [6]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_400bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776649034978024607_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=400, file_path=sample_file_path, tolerance=20)
result

{'latency_ms': 48.938224411010744,
 'model_compute_ms': 42.07099838256836,
 'expert_ffn_ms': 21.17774721980095,
 'attention_ms': 20.89325116276741,
 'matched_rows': 10.0}

In [7]:
# Sample usage with your attached file:
sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/sharegpt_312bs/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776649470343737480_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=312, file_path=sample_file_path, tolerance=20)
result

{'latency_ms': 45.720050511397716,
 'model_compute_ms': 40.24405195581632,
 'expert_ffn_ms': 25.77142891545934,
 'attention_ms': 14.472623040356973,
 'matched_rows': 254.0}

# with vs without config

In [8]:
bs = 16
tolerence = 2

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776984232248404187_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"without config batch size = {bs}, {result}")

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius_after_config_tuning/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776986420540551085_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"with config batch size = {bs}, {result}")

without config batch size = 16, {'latency_ms': 39.76391380809736, 'model_compute_ms': 32.61170809756043, 'expert_ffn_ms': 21.290050657555273, 'attention_ms': 11.32165744000516, 'matched_rows': 1023.0}
with config batch size = 16, {'latency_ms': 32.969817641185735, 'model_compute_ms': 32.03841487706056, 'expert_ffn_ms': 21.282950832016517, 'attention_ms': 10.755464045044041, 'matched_rows': 1026.0}


In [9]:
bs = 48
tolerence = 4

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776984232248404187_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"without config batch size = {bs}, {result}")

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius_after_config_tuning/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776986420540551085_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"with config batch size = {bs}, {result}")

without config batch size = 48, {'latency_ms': 36.146990604251236, 'model_compute_ms': 34.756336904381335, 'expert_ffn_ms': 26.357058515125406, 'attention_ms': 8.399278389255931, 'matched_rows': 766.0}
with config batch size = 48, {'latency_ms': 35.937617633164635, 'model_compute_ms': 34.56098066307458, 'expert_ffn_ms': 26.260075378933553, 'attention_ms': 8.300905284141026, 'matched_rows': 763.0}


In [10]:
bs = 144
tolerence = 5

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776984232248404187_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"without config batch size = {bs}, {result}")

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius_after_config_tuning/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776986420540551085_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"with config batch size = {bs}, {result}")

without config batch size = 144, {'latency_ms': 41.06106961952461, 'model_compute_ms': 37.15778850199102, 'expert_ffn_ms': 23.31580923497677, 'attention_ms': 13.841979267014253, 'matched_rows': 364.0}
with config batch size = 144, {'latency_ms': 43.419568279427544, 'model_compute_ms': 39.49922442369058, 'expert_ffn_ms': 25.279442837036832, 'attention_ms': 14.219781586653749, 'matched_rows': 355.0}


In [11]:
bs = 208
tolerence = 5

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776984232248404187_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"without config batch size = {bs}, {result}")

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius_after_config_tuning/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776986420540551085_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"with config batch size = {bs}, {result}")

without config batch size = 208, {'latency_ms': 38.514879045443294, 'model_compute_ms': 34.06179181267233, 'expert_ffn_ms': 19.077022240172685, 'attention_ms': 14.984769572499651, 'matched_rows': 221.0}
with config batch size = 208, {'latency_ms': 47.507193843020666, 'model_compute_ms': 41.75458914519362, 'expert_ffn_ms': 23.792808780941783, 'attention_ms': 17.96178036425184, 'matched_rows': 237.0}


In [12]:
bs = 320
tolerence = 10

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776984232248404187_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"without config batch size = {bs}, {result}")

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius_after_config_tuning/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776986420540551085_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"with config batch size = {bs}, {result}")

without config batch size = 320, {'latency_ms': 46.94236354933259, 'model_compute_ms': 41.48385689666917, 'expert_ffn_ms': 21.150904760176306, 'attention_ms': 20.33295213649286, 'matched_rows': 181.0}
with config batch size = 320, {'latency_ms': 48.040178111180126, 'model_compute_ms': 42.365877754330015, 'expert_ffn_ms': 21.435929210074825, 'attention_ms': 20.929948544255193, 'matched_rows': 193.0}


In [13]:
bs = 400
tolerence = 10

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776984232248404187_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"without config batch size = {bs}, {result}")

sample_file_path = "/export2/obasit/ClusterMoE/logs/qwen1.5_2.7B/iter_logs/4TP1Prefill_1TP1Decode/sharegpt_cutlass/qwen_disagg_example_2gpu_nebius_after_config_tuning/decode_i0_p8200_g4/iter_profiles/iter_profile_instance_1776986420540551085_rank_0_dp_0_tp_0.jsonl"

result = average_iter_metrics_for_batch(batch_size=bs, file_path=sample_file_path, tolerance=tolerence)
print(f"with config batch size = {bs}, {result}")

without config batch size = 400, {'latency_ms': 52.898699637020336, 'model_compute_ms': 46.126401250502646, 'expert_ffn_ms': 21.645646264272578, 'attention_ms': 24.480754986230064, 'matched_rows': 85.0}
with config batch size = 400, {'latency_ms': 54.23272923060826, 'model_compute_ms': 47.15157972063337, 'expert_ffn_ms': 21.767995391573226, 'attention_ms': 25.383584329060145, 'matched_rows': 14.0}
